In [1]:
!pip -q install opencv-python tqdm scikit-learn

In [2]:
import os
import cv2
import copy
import random
import numpy as np
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

In [4]:
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.lower().endswith((".mp4", ".avi", ".mov", ".mkv", ".webm")):
            print(os.path.join(dirname, filename))

/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/Eating_in_classroom_040.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/Writing_On_Board_076.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/Sitting_on_Desk_016.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/nofi048.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/Holding_Book_084.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/nofi017.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/nofi088.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/Explaining_the_Subject_107.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/nofi138.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/normalclassac

/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/v_c_m_239.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/v_c_m_332.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/v_c_b_62.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/v_c_b_385.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/v_c_b_554.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/v_c_b_566.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/v_c_b_510.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/fi141.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/v_c_m_45.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/v_c_m_147.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/v_c_m_265.mp4
/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom/

In [5]:
NORMAL_ROOT = "/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity"
FIGHT_ROOT  = "/kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom"

print("NORMAL_ROOT exists:", os.path.exists(NORMAL_ROOT))
print("FIGHT_ROOT exists :", os.path.exists(FIGHT_ROOT))

NORMAL_ROOT exists: True
FIGHT_ROOT exists : True


In [6]:
def inspect_root(root, max_files=10):
    print("\nROOT:", root)
    if not os.path.exists(root):
        print("NOT FOUND")
        return
    
    total = 0
    for dirname, _, filenames in os.walk(root):
        vids = [f for f in filenames if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv", ".webm"))]
        if vids:
            print("DIR:", dirname)
            for f in vids[:max_files]:
                print("   ", f)
            total += len(vids)
    print("Total videos found:", total)

inspect_root(NORMAL_ROOT)
inspect_root(FIGHT_ROOT)


ROOT: /kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity


DIR: /kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity
    Eating_in_classroom_040.mp4
    Writing_On_Board_076.mp4
    Sitting_on_Desk_016.mp4
    nofi048.mp4
    Holding_Book_084.mp4
    nofi017.mp4
    nofi088.mp4
    Explaining_the_Subject_107.mp4
    nofi138.mp4
    Writting_on_Textbook_018.mp4
Total videos found: 989

ROOT: /kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom


DIR: /kaggle/input/datasets/sadiamostafa/dataset1/fight_custom/fight_custom
    v_c_m_239.mp4
    v_c_m_332.mp4
    v_c_b_62.mp4
    v_c_b_385.mp4
    v_c_b_554.mp4
    v_c_b_566.mp4
    v_c_b_510.mp4
    fi141.mp4
    v_c_m_45.mp4
    v_c_m_147.mp4
Total videos found: 1382


In [7]:
VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv", ".webm")

def collect_videos(root, label_name):
    samples = []
    for dirname, _, filenames in os.walk(root):
        for filename in filenames:
            if filename.lower().endswith(VIDEO_EXTS):
                samples.append({
                    "path": os.path.join(dirname, filename),
                    "label": label_name
                })
    return samples

normal_samples = collect_videos(NORMAL_ROOT, "normal")
fight_samples  = collect_videos(FIGHT_ROOT, "fight")

samples = normal_samples + fight_samples

print("Total samples:", len(samples))
print("Label counts:", Counter([x["label"] for x in samples]))
print("\nFirst 10 samples:")
for s in samples[:10]:
    print(s)

Total samples: 2371
Label counts: Counter({'fight': 1382, 'normal': 989})

First 10 samples:
{'path': '/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/Eating_in_classroom_040.mp4', 'label': 'normal'}
{'path': '/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/Writing_On_Board_076.mp4', 'label': 'normal'}
{'path': '/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/Sitting_on_Desk_016.mp4', 'label': 'normal'}
{'path': '/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/nofi048.mp4', 'label': 'normal'}
{'path': '/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/Holding_Book_084.mp4', 'label': 'normal'}
{'path': '/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassactivity/nofi017.mp4', 'label': 'normal'}
{'path': '/kaggle/input/datasets/sadiamostafa/dataset1/normalclassactivity/normalclassact

In [8]:
labels = [s["label"] for s in samples]

train_samples, temp_samples = train_test_split(
    samples,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

temp_labels = [s["label"] for s in temp_samples]

val_samples, test_samples = train_test_split(
    temp_samples,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

print("Train:", len(train_samples), Counter([x["label"] for x in train_samples]))
print("Val  :", len(val_samples), Counter([x["label"] for x in val_samples]))
print("Test :", len(test_samples), Counter([x["label"] for x in test_samples]))

Train: 1659 Counter({'fight': 967, 'normal': 692})
Val  : 356 Counter({'fight': 207, 'normal': 149})
Test : 356 Counter({'fight': 208, 'normal': 148})


In [9]:
LABEL_MAP = {
    "normal": 0,
    "fight": 1
}

IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
print(LABEL_MAP)

{'normal': 0, 'fight': 1}


In [10]:
class VideoAugment:
    def __init__(self, img_size=224):
        self.img_size = img_size
        self.color_jitter = transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.15,
            hue=0.03
        )
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def horizontal_flip_clip(self, frames):
        if random.random() < 0.5:
            frames = [cv2.flip(f, 1) for f in frames]
        return frames

    def random_crop_resize_clip(self, frames):
        if random.random() < 0.5:
            h, w, _ = frames[0].shape
            scale = random.uniform(0.85, 1.0)
            new_h = int(h * scale)
            new_w = int(w * scale)

            top = random.randint(0, h - new_h) if h > new_h else 0
            left = random.randint(0, w - new_w) if w > new_w else 0

            out = []
            for f in frames:
                c = f[top:top + new_h, left:left + new_w]
                c = cv2.resize(c, (self.img_size, self.img_size))
                out.append(c)
            return out
        return frames

    def add_gaussian_noise_clip(self, frames):
        if random.random() < 0.25:
            out = []
            for f in frames:
                noise = np.random.normal(0, 5, f.shape).astype(np.float32)
                x = np.clip(f.astype(np.float32) + noise, 0, 255).astype(np.uint8)
                out.append(x)
            return out
        return frames

    def to_tensor_clip(self, frames, train=False):
        out = []
        for f in frames:
            img = transforms.ToPILImage()(f)
            if train:
                img = self.color_jitter(img)
            t = transforms.ToTensor()(img)
            t = self.normalize(t)
            out.append(t)
        return torch.stack(out)

    def __call__(self, frames, train=False):
        if train:
            frames = self.horizontal_flip_clip(frames)
            frames = self.random_crop_resize_clip(frames)
            frames = self.add_gaussian_noise_clip(frames)
        return self.to_tensor_clip(frames, train=train)

In [11]:
class FightBinaryDataset(Dataset):
    def __init__(self, samples, num_frames=16, img_size=224, train=False):
        self.samples = samples
        self.num_frames = num_frames
        self.img_size = img_size
        self.train = train
        self.augment = VideoAugment(img_size=img_size)

    def __len__(self):
        return len(self.samples)

    def _blank_clip(self):
        blank = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        frames = [blank.copy() for _ in range(self.num_frames)]
        return self.augment(frames, train=self.train)

    def _read_frames_uniform(self, video_path):
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            return self._blank_clip()

        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count <= 0:
            cap.release()
            return self._blank_clip()

        idxs = np.linspace(0, max(0, frame_count - 1), self.num_frames).astype(int)
        idxs_set = set(idxs.tolist())

        frames = []
        i = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if i in idxs_set:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (self.img_size, self.img_size))
                frames.append(frame)
            i += 1

        cap.release()

        if len(frames) == 0:
            return self._blank_clip()

        while len(frames) < self.num_frames:
            frames.append(frames[-1].copy())

        frames = frames[:self.num_frames]
        x = self.augment(frames, train=self.train)
        return x

    def __getitem__(self, idx):
        item = self.samples[idx]
        x = self._read_frames_uniform(item["path"])
        y = torch.tensor(LABEL_MAP[item["label"]], dtype=torch.long)
        return x, y

In [12]:
train_ds = FightBinaryDataset(train_samples, num_frames=16, img_size=224, train=True)
val_ds   = FightBinaryDataset(val_samples,   num_frames=16, img_size=224, train=False)
test_ds  = FightBinaryDataset(test_samples,  num_frames=16, img_size=224, train=False)

print("Train:", len(train_ds))
print("Val  :", len(val_ds))
print("Test :", len(test_ds))

Train: 1659
Val  : 356
Test : 356


In [13]:
train_labels = [LABEL_MAP[s["label"]] for s in train_samples]
class_counts = Counter(train_labels)
print("Train class counts:", class_counts)

sample_weights = [1.0 / class_counts[label] for label in train_labels]
sample_weights = torch.DoubleTensor(sample_weights)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

BATCH_SIZE = 8
NUM_WORKERS = 2
PIN_MEMORY = True

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

Train class counts: Counter({1: 967, 0: 692})


In [14]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda


GPU: Tesla T4


In [15]:
class MobileNetV3GRUFight(nn.Module):
    def __init__(self, hidden_size=256, num_layers=1, bidirectional=True, dropout=0.3):
        super().__init__()

        try:
            base = models.mobilenet_v3_large(
                weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
            )
            print("Loaded ImageNet pretrained MobileNetV3.")
        except Exception as e:
            print("Could not load ImageNet pretrained weights. Using random init.")
            print("Reason:", e)
            base = models.mobilenet_v3_large(weights=None)

        self.backbone = base.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.feature_dim = 960
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional
        self.num_dirs = 2 if bidirectional else 1

        self.gru = nn.GRU(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=0.0 if num_layers == 1 else dropout
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_size * self.num_dirs, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        # x: [B, T, C, H, W]
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)

        f = self.backbone(x)
        f = self.pool(f).flatten(1)
        f = f.view(B, T, self.feature_dim)

        out, _ = self.gru(f)
        seq_feat = out[:, -1, :]

        logits = self.head(seq_feat)
        return logits

In [16]:
model = MobileNetV3GRUFight(
    hidden_size=256,
    num_layers=1,
    bidirectional=True,
    dropout=0.3
).to(device)

print(model.__class__.__name__)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


  0%|          | 0.00/21.1M [00:00<?, ?B/s]

 48%|████▊     | 10.1M/21.1M [00:00<00:00, 106MB/s]

100%|██████████| 21.1M/21.1M [00:00<00:00, 148MB/s]

Loaded ImageNet pretrained MobileNetV3.


MobileNetV3GRUFight


In [17]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [18]:
def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()

    losses = []
    all_true, all_pred = [], []

    for x, y in tqdm(loader, leave=False):
        x = x.to(device)
        y = y.to(device)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        losses.append(loss.item())

        pred = logits.argmax(dim=1)
        all_true.extend(y.detach().cpu().numpy().tolist())
        all_pred.extend(pred.detach().cpu().numpy().tolist())

    metrics = {
        "loss": float(np.mean(losses)),
        "acc": accuracy_score(all_true, all_pred),
        "f1_macro": f1_score(all_true, all_pred, average="macro"),
        "f1_fight": f1_score(all_true, all_pred, average=None, labels=[1])[0],
        "cm": confusion_matrix(all_true, all_pred),
        "true": all_true,
        "pred": all_pred,
    }
    return metrics

In [19]:
best_score = -1
best_state = None

best_path = "/kaggle/working/best_fight_binary_mobilenetv3_gru.pth"

EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader, optimizer=None)

    score = val_metrics["f1_fight"] + val_metrics["f1_macro"]
    scheduler.step(score)

    print(f"\nEpoch {epoch:02d}")
    print(f"Train | loss {train_metrics['loss']:.4f} | acc {train_metrics['acc']:.4f} | macro_f1 {train_metrics['f1_macro']:.4f} | fight_f1 {train_metrics['f1_fight']:.4f}")
    print(f"Val   | loss {val_metrics['loss']:.4f} | acc {val_metrics['acc']:.4f} | macro_f1 {val_metrics['f1_macro']:.4f} | fight_f1 {val_metrics['f1_fight']:.4f}")
    print("Val CM:\n", val_metrics["cm"])

    if score > best_score:
        best_score = score
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, best_path)
        print("✅ Saved best model.")

print("\nBest score:", best_score)
print("Saved to:", best_path)

  0%|          | 0/208 [00:00<?, ?it/s]

  0%|          | 1/208 [00:16<57:25, 16.65s/it]

  1%|          | 2/208 [00:16<24:10,  7.04s/it]

  1%|▏         | 3/208 [00:25<26:03,  7.63s/it]

  2%|▏         | 4/208 [00:25<16:07,  4.74s/it]

...

 60%|█████▉    | 124/208 [14:56<11:01,  7.88s/it]

 60%|██████    | 125/208 [14:56<07:45,  5.61s/it]

 61%|██████    | 126/208 [15:04<08:28,  6.20s/it]

 61%|██████    | 127/208 [15:04<05:59,  4.44s/it]

 62%|██████▏   | 128/208 [15:28<13:47, 10.34s/it]

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_metrics = run_epoch(model, test_loader, optimizer=None)

print("\n===== TEST RESULTS =====")
print(f"Test | loss {test_metrics['loss']:.4f} | acc {test_metrics['acc']:.4f} | macro_f1 {test_metrics['f1_macro']:.4f} | fight_f1 {test_metrics['f1_fight']:.4f}")

print("\nConfusion Matrix:")
print(test_metrics["cm"])

print("\nClassification Report:")
print(classification_report(
    test_metrics["true"],
    test_metrics["pred"],
    target_names=["normal", "fight"],
    digits=4
))

In [ ]:
summary_path = "/kaggle/working/fight_binary_summary.txt"

with open(summary_path, "w") as f:
    f.write(f"Best score: {best_score}\n")
    f.write(f"Model path: {best_path}\n")
    f.write(f"Train size: {len(train_samples)}\n")
    f.write(f"Val size: {len(val_samples)}\n")
    f.write(f"Test size: {len(test_samples)}\n")
    f.write(f"Train counts: {Counter([x['label'] for x in train_samples])}\n")
    f.write(f"Val counts: {Counter([x['label'] for x in val_samples])}\n")
    f.write(f"Test counts: {Counter([x['label'] for x in test_samples])}\n")

print("Saved summary:", summary_path)

In [ ]:
IMG_SIZE = 224
NUM_FRAMES = 16

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

def load_video_clip(video_path, num_frames=16, img_size=224):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        raise ValueError(f"No frames found in: {video_path}")

    idxs = np.linspace(0, max(0, frame_count - 1), num_frames).astype(int)
    idxs_set = set(idxs.tolist())

    frames = []
    i = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if i in idxs_set:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (img_size, img_size))
            frames.append(frame)
        i += 1

    cap.release()

    if len(frames) == 0:
        raise ValueError("No frames extracted")

    while len(frames) < num_frames:
        frames.append(frames[-1].copy())

    frames = frames[:num_frames]

    clip = []
    for f in frames:
        t = transforms.ToTensor()(f)
        t = normalize(t)
        clip.append(t)

    clip = torch.stack(clip).unsqueeze(0)
    return clip, frames

def predict_video(video_path, threshold=0.75):
    x, frames = load_video_clip(video_path, num_frames=NUM_FRAMES, img_size=IMG_SIZE)
    x = x.to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    p_normal = float(probs[0])
    p_fight = float(probs[1])

    if p_fight >= threshold:
        final_label = "fight"
    elif p_fight >= 0.50:
        final_label = "suspicious"
    else:
        final_label = "normal"

    return {
        "final_label": final_label,
        "p_normal": p_normal,
        "p_fight": p_fight,
        "frames": frames
    }

In [ ]:
import os

print("Files in /kaggle/working:")
for f in os.listdir("/kaggle/working"):
    print(f)

In [ ]:
import torch

path = "/kaggle/working/best_fight_binary_mobilenetv3_gru.pth"
print("Exists:", os.path.exists(path))
if os.path.exists(path):
    ckpt = torch.load(path, map_location="cpu")
    print("Loaded checkpoint type:", type(ckpt))